# Импортируем все необходимые библиотеки


In [1]:
%%capture
!pip install pytorch-lightning

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


import torch

from transformers import T5Tokenizer
from transformers import T5ForConditionalGeneration
from torch.optim import AdamW

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

pl.seed_everything(100)

import warnings
warnings.filterwarnings("ignore")

INFO:lightning_fabric.utilities.seed:Seed set to 100


# Импортируем датасет

squad - классический набор данных на huggingface, содержащий статьи с Wikipedia (контекст) + вопорос + ответ

In [3]:
# Определяем пути к обучающим и валидационным файлам в виде словаря
# 'train' указывает на обучающий набор данных, 'validation' на валидационный
splits = {'train': 'plain_text/train-00000-of-00001.parquet', 'validation': 'plain_text/validation-00000-of-00001.parquet'}

# Загружаем обучающий набор данных из SQuAD (через Hugging Face datasets) в DataFrame pandas
# Используем pd.read_parquet для чтения файла формата Parquet
df = pd.read_parquet("hf://datasets/rajpurkar/squad/" + splits["train"])

# Закомментированная строка ниже показывает альтернативный способ загрузки данных из CSV-файла
# df = pd.read_csv("/kaggle/input/question-answer-dataset-qa/train.csv")

# Выводим названия всех столбцов загруженного DataFrame, чтобы понять его структуру
df.columns

Index(['id', 'title', 'context', 'question', 'answers'], dtype='object')

In [4]:
df.head()

,id,title,context,question,answers
0,5733be284776f41900661182,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,"{'text': ['Saint Bernadette Soubirous'], 'answ..."
1,5733be284776f4190066117f,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,"{'text': ['a copper statue of Christ'], 'answe..."
2,5733be284776f41900661180,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,"{'text': ['the Main Building'], 'answer_start'..."
3,5733be284776f41900661181,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,{'text': ['a Marian place of prayer and reflec...
4,5733be284776f4190066117e,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,{'text': ['a golden statue of the Virgin Mary'...


In [5]:
# extract answer text from complex obects inside "answers" column and use the to fill a new column named "answer_text"
df['answer_text'] = df['answers'].apply(lambda x: x['text'][0])

In [6]:
df.head(1)

,id,title,context,question,answers,answer_text
0,5733be284776f41900661182,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,"{'text': ['Saint Bernadette Soubirous'], 'answ...",Saint Bernadette Soubirous


In [7]:
# выбираем только нужные столбцы
df = df[['context','question', 'answer_text']]

In [8]:
print("Number of records: ", df.shape[0])

Number of records:  87599


# Решаемая проблема: QA (вопросно-ответная система)

<hr>

#### "Создаем модель, которая на основе контекста и вопроса, формулирует ответ"

Например,

*Контекст = "Clustering groups of similar cases, for example, can find similar patients, or can be used for customer segmentation in the banking field. Association technique is used for finding items or events that often co-occur, for example, grocery items that are usually bought together by a particular customer. Anomaly detection is used to discover abnormal
and unusual cases, for example, it is used for credit card fraud detection."*

*Вопрос = "what is the example of Anomaly detection?"*

*Ответ =  ????????????????????????????????*


In [9]:
# приводим к нижнему регистру
df["context"] = df["context"].str.lower()
df["question"] = df["question"].str.lower()
df["answer_text"] = df["answer_text"].str.lower()

df.head()

,context,question,answer_text
0,"architecturally, the school has a catholic cha...",to whom did the virgin mary allegedly appear i...,saint bernadette soubirous
1,"architecturally, the school has a catholic cha...",what is in front of the notre dame main building?,a copper statue of christ
2,"architecturally, the school has a catholic cha...",the basilica of the sacred heart at notre dame...,the main building
3,"architecturally, the school has a catholic cha...",what is the grotto at notre dame?,a marian place of prayer and reflection
4,"architecturally, the school has a catholic cha...",what sits on top of the main building at notre...,a golden statue of the virgin mary



# Инициализация параметров

<hr>


**Длина входного примера:** Это количество входных лексем (например, слов или символов) в одном примере, который подается в модель во время обучения. Например, если вы обучаете языковую модель предсказывать следующее слово в предложении, длина входных данных будет равна количеству слов в предложении.

**Длина выходного примера:** Это количество выходных лексем (например, слов или символов) в одном примере, который модель должна сгенерировать в процессе обучения. Длина вывода - это количество слов в предложении, которое модель должна предсказать.

**Размер батча при обучении**

**Размер валидационного батча**

**Кол-во эпох:** Эпоха означает один проход по всему обучающему набору данных. Так, если набор обучающих данных содержит 1000 примеров, а размер батча равен 32, то для завершения одной эпохи потребуется 32 шага обучения. Если модель обучается в течение 10 эпох, то всего будет обработано 10 * 1000 = 10000 примеров.

In [10]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_MAX_LEN = 512 # Input length
OUT_MAX_LEN = 128 # Output Length for Decoder
TRAIN_BATCH_SIZE = 8 # Training Batch Size
VALID_BATCH_SIZE = 2 # Validation Batch Size
EPOCHS = 5 # Number of Iterations

# T5 Transformer

<hr>


Статья по T5: https://arxiv.org/pdf/1910.10683.pdf

| Model Name | Parameters | Encoder Layers | Decoder Layers | Hidden Size (d_model) | FFN Size (d_ff) | Attention Heads | Max Input Length | Max Output Length |
|------------|------------|----------------|----------------|------------------------|-----------------|------------------|------------------|-------------------|
| T5-Small   | ~60M       | 6              | 6              | 512                    | 2048            | 8                | 512 tokens       | 512 tokens        |
| T5-Base    | ~220M      | 12             | 12             | 768                    | 3072            | 12               | 512 tokens       | 512 tokens        |
| T5-Large   | ~770M      | 24             | 24             | 1024                   | 4096            | 16               | 512 tokens       | 512 tokens        |
| T5-3B      | ~3B        | 24             | 24             | 1024                   | 16384           | 32               | 512 tokens       | 512 tokens        |
| T5-11B     | ~11B       | 24             | 24             | 1024                   | 65536           | 128              | 512 tokens       | 512 tokens        |



* **Context size**: Standard T5 models are trained with a 512-token encoder input and 512-token decoder output.
* **Encoder–decoder architecture**: Input and output lengths are separate, unlike decoder-only models (e.g., GPT).
    * Encoder input length and decoder output length can differ
    * T5 uses relative position bias, not absolute positional embeddings
    * This means there is no fixed maximum sequence length baked into the model weights
* **Relative positional embeddings**: T5 uses relative position bias, which technically allows longer contexts if retrained or fine-tuned accordingly.
* **Extended variants**: Models like Long-T5 increase input context (e.g., 4k–16k tokens), but those are not standard T5.

In [11]:
MODEL_NAME = "t5-base" # https://huggingface.co/google-t5/t5-base

# initiate tokenizer for T5
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, model_max_length= INPUT_MAX_LEN)

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
print("eos_token: {} and id: {}".format(tokenizer.eos_token, tokenizer.eos_token_id)) # End of token (eos_token)
print("unk_token: {} and id: {}".format(tokenizer.unk_token, tokenizer.eos_token_id)) # Unknown token (unk_token)
print("pad_token: {} and id: {}".format(tokenizer.pad_token, tokenizer.eos_token_id)) # Pad token (pad_token)

eos_token: </s> and id: 1
unk_token: <unk> and id: 1
pad_token: <pad> and id: 1


# Подготовка датасета

<hr>

In [13]:
class T5Dataset:
    """
    Custom PyTorch-style dataset for training a T5 model on
    context-question-answer style data.
    """

    def __init__(self, context, question, target):
        # Store raw data lists
        self.context = context
        self.question = question
        self.target = target
        # Tokenizer used for encoding text
        self.tokenizer = tokenizer
        # Maximum sequence lengths for inputs and outputs
        self.input_max_len = INPUT_MAX_LEN
        self.out_max_len = OUT_MAX_LEN

    def __len__(self):
        """
        Returns the total number of samples in the dataset.
        """
        return len(self.context)

    def __getitem__(self, item):
        """
        Returns one training example at the given index.
        """
        # Get and clean the context text
        context = str(self.context[item])
        context = " ".join(context.split())

        # Get and clean the question text
        question = str(self.question[item])
        question = " ".join(question.split())

        # Get and clean the target (answer) text
        target = str(self.target[item])
        target = " ".join(target.split())

        # Tokenize context and question together (encoder input)
        inputs_encoding = self.tokenizer(
            context,
            question,
            add_special_tokens=True,
            max_length=self.input_max_len,
            padding = 'max_length',        # Pad sequences to max length
            truncation='only_first',       # Truncate context if too long
            return_attention_mask=True,
            return_tensors="pt"            # Return PyTorch tensors
        )

        # Tokenize the target/answer (decoder input)
        output_encoding = self.tokenizer(
            target,
            None,
            add_special_tokens=True,
            max_length=self.out_max_len,
            padding = 'max_length',      # Pad sequences to max length
            truncation= True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        # Flatten input IDs and attention masks for model compatibility
        inputs_ids = inputs_encoding["input_ids"].flatten()
        attention_mask = inputs_encoding["attention_mask"].flatten()
        # Extract target token IDs
        labels = output_encoding["input_ids"]

        # Replace padding token IDs (0) with -100 so loss ignores them
        # (required by T5 / HuggingFace loss computation)
        labels[labels == 0] = -100  # As per T5 Documentation

        # Flatten labels tensor
        labels = labels.flatten()

        out = {
            "context": context,
            "question": question,
            "answer": target,
            "inputs_ids": inputs_ids,
            "attention_mask": attention_mask,
            "targets": labels
        }


        return out

# DataLoader

<hr>


In [14]:
class T5DatasetModule(pl.LightningDataModule):
    """
    LightningDataModule для подготовки данных для модели T5.
    Обертывает T5Dataset и предоставляет DataLoader'ы для обучения и валидации.
    """

    def __init__(self, df_train, df_valid):
        super().__init__()
        # Сохраняем DataFrame'ы для обучения и валидации
        self.df_train = df_train
        self.df_valid = df_valid
        # Используем существующий токенизатор
        self.tokenizer = tokenizer
        # Максимальные длины входной и выходной последовательностей
        self.input_max_len = INPUT_MAX_LEN
        self.out_max_len = OUT_MAX_LEN


    def setup(self, stage=None):
        """
        Подготавливает датасеты для обучения и валидации, создавая экземпляры T5Dataset.
        """
        self.train_dataset = T5Dataset(
        context=self.df_train.context.values,
        question=self.df_train.question.values,
        target=self.df_train.answer_text.values
        )

        self.valid_dataset = T5Dataset(
        context=self.df_valid.context.values,
        question=self.df_valid.question.values,
        target=self.df_valid.answer_text.values
        )

    def train_dataloader(self):
        """
        Создает DataLoader для обучающего набора данных.
        """
        return torch.utils.data.DataLoader(
         self.train_dataset,
         batch_size= TRAIN_BATCH_SIZE,
         shuffle=True,
         num_workers=4
        )


    def val_dataloader(self):
        """
        Создает DataLoader для валидационного набора данных.
        """
        return torch.utils.data.DataLoader(
         self.valid_dataset,
         batch_size= VALID_BATCH_SIZE,
         num_workers=1
        )

# Сама модель

<hr>

In [15]:
class T5Model(pl.LightningModule):
    """
    LightningModule, инкапсулирующий модель T5 для задач вопросно-ответных систем.
    """

    def __init__(self):
        super().__init__()
        # Импорт предварительно обученной модели T5 для условной генерации из transformers
        self.model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME, return_dict=True)

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Прямой проход модели T5. Принимает ID входных токенов, маску внимания и опционально метки.
        Возвращает потерю (если предоставлены метки) и логиты.
        """
        output = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        return output.loss, output.logits


    def training_step(self, batch, batch_idx):
        """
        Шаг обучения: вычисляет потерю и логирует ее.
        """
        input_ids = batch["inputs_ids"]
        attention_mask = batch["attention_mask"]
        labels= batch["targets"]
        loss, outputs = self(input_ids, attention_mask, labels)

        # Логирование тренировочной потери
        self.log("train_loss", loss, prog_bar=True, logger=True)

        return loss

    def validation_step(self, batch, batch_idx):
        """
        Шаг валидации: вычисляет потерю на валидационном наборе и логирует ее.
        """
        input_ids = batch["inputs_ids"]
        attention_mask = batch["attention_mask"]
        labels= batch["targets"]
        loss, outputs = self(input_ids, attention_mask, labels)

        # Логирование валидационной потери
        self.log("val_loss", loss, prog_bar=True, logger=True)

        return loss


    def configure_optimizers(self):
        """
        Конфигурация оптимизатора. Использует AdamW с заданной скоростью обучения.
        """
        return AdamW(self.parameters(), lr=0.0001)

# Обучение

<hr>

In [18]:
def run():
    # Разделяем данные на обучающую и валидационную выборки
    df_train, df_valid = train_test_split(
        df[0:10000], test_size=0.2, random_state=101
    )

    # Заполняем пропущенные значения строкой "none"
    df_train = df_train.fillna("none")
    df_valid = df_valid.fillna("none")

    # Удаляем лишние пробелы в текстовых колонках
    df_train['context'] = df_train['context'].apply(lambda x: " ".join(x.split()))
    df_valid['context'] = df_valid['context'].apply(lambda x: " ".join(x.split()))

    df_train['answer_text'] = df_train['answer_text'].apply(lambda x: " ".join(x.split()))
    df_valid['answer_text'] = df_valid['answer_text'].apply(lambda x: " ".join(x.split()))

    df_train['question'] = df_train['question'].apply(lambda x: " ".join(x.split()))
    df_valid['question'] = df_valid['question'].apply(lambda x: " ".join(x.split()))

    # Сбрасываем индексы DataFrame'ов
    df_train = df_train.reset_index(drop=True)
    df_valid = df_valid.reset_index(drop=True)

    # Инициализируем модуль данных для PyTorch Lightning
    dataModule = T5DatasetModule(df_train, df_valid)
    dataModule.setup()

    # Определяем устройство для обучения (GPU или CPU)
    device = DEVICE
    # Инициализируем модель T5
    models = T5Model()
    models.to(device)

    # Настраиваем колбэк для сохранения лучшей модели
    checkpoint_callback  = ModelCheckpoint(
        dirpath="/kaggle/working", # Директория для сохранения чекпоинтов
        filename="best_checkpoint", # Имя файла чекпоинта
        save_top_k=2, # Сохранять 2 лучшие модели
        verbose=True, # Выводить информацию при сохранении
        monitor="val_loss", # Метрика для мониторинга (валидационная потеря)
        mode="min" # Режим мониторинга: искать минимальное значение метрики
    )

    # Инициализируем трейнер PyTorch Lightning
    trainer = pl.Trainer(
        callbacks = checkpoint_callback, # Передаем колбэк для сохранения модели
        max_epochs= EPOCHS, # Максимальное количество эпох обучения
        devices=1, # Использовать 1 устройство (GPU)
        accelerator="gpu" # Использовать GPU в качестве акселератора
    )

    # Запускаем обучение модели
    trainer.fit(models, dataModule)

In [19]:
# use GPU in Google Colab for faster training here!
run()

Epoch 4/4  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1000/1000 0:21:49 • 0:00:00 0.77it/s v_num: 0.000 train_loss: 0.039    
                                                                                 val_loss: 0.309                   

INFO:pytorch_lightning.utilities.rank_zero:Epoch 4, global step 5000: 'val_loss' was not in top 2
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.



# Инференс модели

<hr>

In [ ]:
# train_model = T5Model.load_from_checkpoint("/kaggle/working/best_checkpoint-v1.ckpt")

train_model.freeze()

def generate_question(context, question):
    """
    Генерирует ответ на вопрос на основе предоставленного контекста с использованием обученной модели T5.

    Args:
        context (str): Текстовый контекст, из которого генерируется ответ.
        question (str): Вопрос, на который нужно ответить.

    Returns:
        str: Сгенерированный ответ.
    """

    # Токенизируем контекст и вопрос вместе для входных данных модели
    inputs_encoding  = tokenizer(
        context,
        question,
        add_special_tokens=True,
        max_length= INPUT_MAX_LEN,
        padding = 'max_length',
        truncation='only_first',
        return_attention_mask=True,
        return_tensors="pt"
        )

    # Генерируем ID токенов ответа с помощью модели
    generate_ids = train_model.model.generate(
        input_ids = inputs_encoding["input_ids"],
        attention_mask = inputs_encoding["attention_mask"],
        max_length = INPUT_MAX_LEN,
        num_beams = 4, # Детерминированный поиск луча с 4 лучами (beam_search)
        num_return_sequences = 1,
        no_repeat_ngram_size=2,
        early_stopping=True,
        )

    # Декодируем сгенерированные ID токенов обратно в текст
    preds = [
        tokenizer.decode(gen_id,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True)
        for gen_id in generate_ids
    ]

    return "".join(preds)


In [ ]:
context = "Clustering groups of similar cases, for example, \
can find similar patients, or can be used for customer segmentation in the \
banking field. Association technique is used for finding items or events that \
often co-occur, for example, grocery items that are usually bought together\
by a particular customer. Anomaly detection is used to discover abnormal \
and unusual cases, for example, it is used for credit card fraud detection."

que = "what is the example of Anomaly detection?"

# Генерируем ответ на вопрос с использованием контекста
print(generate_question(context, que))


credit card fraud detection


In [ ]:
context = "Classification is used when your target is categorical, while regression is used when your target variable\
is continuous. Both classification and regression belong to the category of supervised machine learning algorithms."

que = "When is classification used?"

# Генерируем ответ на вопрос с использованием контекста
print(generate_question(context, que))


when your target is categorical
